# BenchFlow Scene Patterns

Three evaluation patterns — baseline, self-review (multi-turn), coder-reviewer (multi-round) — run end-to-end with `bf.run()`.

| Term | Definition | Example |
|------|-----------|--------|
| **Turn** | One prompt in one ACP session | Coder writes code |
| **Multi-turn** | Same role, multiple turns | Self-review: agent → agent |
| **Round** | One A→B exchange between roles | Coder → Reviewer |
| **Multi-round** | Different roles exchanging turns | Coder → Reviewer → Coder |
| **Scene** | Interaction region with roles + turns | A code-review scene |
| **Trial** | Sequence of scenes in a shared sandbox | Skill-gen → Solve |

**Prerequisites:**
- `pip install benchflow`
- `GEMINI_API_KEY` or `GOOGLE_API_KEY` set
- `DAYTONA_API_KEY` set (or Docker daemon running for `--env docker`)
- A task directory (this notebook uses `.ref/terminal-bench-2/regex-log`)

In [1]:
import logging
from pathlib import Path

import benchflow as bf
from benchflow.trial import Role, Scene, TrialConfig, Turn

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(message)s")

TASK = Path(".ref/terminal-bench-2/regex-log")
ENV = "daytona"
AGENT = "gemini"
MODEL = "gemini-3.1-flash-lite-preview"

results = {}  # pattern_name -> RunResult
print(f"Task: {TASK}  Env: {ENV}  Agent: {AGENT}  Model: {MODEL}")

Task: .ref/terminal-bench-2/regex-log  Env: daytona  Agent: gemini  Model: gemini-3.1-flash-lite-preview


## Pattern 1: Single-Agent Baseline

One agent, one turn. The simplest configuration.

In [2]:
config = TrialConfig(
    task_path=TASK,
    scenes=[Scene.single(agent=AGENT, model=MODEL)],
    environment=ENV,
)

result = await bf.run(config)
results["baseline"] = result

reward = (result.rewards or {}).get("reward")
print(f"Reward:     {reward}")
print(f"Tool calls: {result.n_tool_calls}")
print(f"Error:      {result.error}")

Reward:     1.0
Tool calls: 3
Error:      None


## Pattern 2: Multi-Turn Self-Review

Same agent gets a second turn to re-examine its own work. This is **multi-turn** — one role, multiple prompts.

Use when: a second pass catches what the first missed. Cost: 2x baseline.

In [3]:
config = TrialConfig(
    task_path=TASK,
    scenes=[Scene(
        name="self-review",
        roles=[Role("agent", AGENT, MODEL)],
        turns=[
            Turn("agent"),
            Turn("agent", "Review your solution. Check edge cases, "
                 "boundary conditions, and adherence to the requirements. Fix issues."),
        ],
    )],
    environment=ENV,
)

result = await bf.run(config)
results["self_review"] = result

reward = (result.rewards or {}).get("reward")
print(f"Reward:     {reward}")
print(f"Tool calls: {result.n_tool_calls}")
print(f"Error:      {result.error}")

Reward:     1.0
Tool calls: 7
Error:      None


## Pattern 3: Coder + Reviewer (Multi-Round)

Two roles — coder and reviewer — in a shared sandbox. This is **multi-round** — different roles exchanging turns.

Communication uses the **outbox convention**:
1. Reviewer writes feedback to `/app/.outbox/coder.json` as `{"to": "coder", "content": "..."}`
2. Scheduler reads the outbox after the reviewer's turn
3. Feedback is automatically injected into the coder's next prompt

Use when: tasks benefit from independent review. Cost: 3x baseline.

In [4]:
config = TrialConfig(
    task_path=TASK,
    scenes=[Scene(
        name="code-review",
        roles=[
            Role("coder", AGENT, MODEL),
            Role("reviewer", AGENT, MODEL),
        ],
        turns=[
            Turn("coder"),
            Turn("reviewer",
                 "Review the code in /app/. Read /app/instruction.md for requirements. "
                 "Check for correctness, edge cases, and missed requirements. "
                 "Write your feedback to /app/.outbox/coder.json as: "
                 '{"to": "coder", "content": "Your specific feedback here."}'),
            Turn("coder",
                 "Read the reviewer's feedback and fix the issues. "
                 "Focus only on what was flagged — don't start over."),
        ],
    )],
    environment=ENV,
)

result = await bf.run(config)
results["coder_reviewer"] = result

reward = (result.rewards or {}).get("reward")
print(f"Reward:     {reward}")
print(f"Tool calls: {result.n_tool_calls}")
print(f"Error:      {result.error}")

Reward:     0.0
Tool calls: 13
Error:      None


## Comparison

In [5]:
print(f"{'Pattern':<20} {'Reward':>8} {'Tools':>6} {'Error'}")
print("-" * 60)
for name, r in results.items():
    reward = (r.rewards or {}).get("reward", "—")
    err = r.error or "—"
    if len(err) > 30:
        err = err[:27] + "..."
    print(f"{name:<20} {reward!s:>8} {r.n_tool_calls:>6} {err}")

Pattern                Reward  Tools Error
------------------------------------------------------------
baseline                  1.0      3 —
self_review               1.0      7 —
coder_reviewer            0.0     13 —


## Mapping to Harbor PR #1462 Concepts

For users coming from Harbor's multi-turn proposal:

| Harbor (PR #1462) | BenchFlow 0.3 | Status |
|-------------------|---------------|--------|
| `BaseUser` | `Role` — any role can be a user, reviewer, or custom agent | **Shipped** |
| `User.run() → str` | `Turn` with a prompt — each turn sends one prompt to one role | **Shipped** |
| Per-round message passing | Outbox files + scheduler injection into next role's prompt | **Shipped** |
| Per-round archiving | `scene_messages.jsonl` in trial directory | **Shipped** |
| `--user` CLI flag | YAML scene config (`scenes:` key in trial config) | **Shipped** |
| `User.run() → None` (stop) | Fixed turn count only — no dynamic termination | **Gap** |
| Oracle access (`/solution`) | Not available to user roles during setup | **Gap** |
| Inter-round verification | `verify()` runs once after all scenes | **Gap** |
| User inspects trajectory | User role cannot read prior agent trajectory between rounds | **Gap** |

**What BenchFlow 0.3 delivers:** Multi-role scenes with outbox messaging, message persistence,
and the same API for single-agent and multi-agent runs. N roles per scene, sequential multi-scene trials.

**What it does NOT deliver yet:** Dynamic termination (user decides when to stop), oracle access
for user roles, per-round verification, or inter-round trajectory inspection. These require
extending the Scene scheduler with callbacks — tracked for 0.4.